# 🟡 Medium: Softmax Cross-Entropy + Gradient (NumPy)

Implement softmax cross-entropy **and its gradient** in one pass, using only NumPy.

### Core Idea

Softmax turns logits into probabilities, cross-entropy scores them:

$$p_{ij} = \frac{e^{z_{ij}}}{\sum_k e^{z_{ik}}}, \qquad \mathcal{L} = -\frac{1}{N}\sum_i \log p_{i, y_i}$$

**Why they are always fused.** Softmax alone has a full $C \times C$ Jacobian
$\partial p_i / \partial z_j = p_i(\delta_{ij} - p_j)$ — ugly and expensive. Compose it with
cross-entropy and almost everything cancels, leaving the most elegant gradient in deep learning:

$$\frac{\partial \mathcal{L}}{\partial z_{ij}} = \frac{p_{ij} - \mathbb{1}[j = y_i]}{N}$$

*"predicted minus actual"* — that is the entire backward pass. This is why every framework ships
`cross_entropy(logits, labels)` instead of `nll_loss(softmax(logits))`: it is faster **and** it never
computes `log(0)`.

**Numerical stability.** `exp(1000)` overflows to `inf` in float64. But softmax is shift-invariant —
$\mathrm{softmax}(z) = \mathrm{softmax}(z - c)$ for any $c$ — so subtract the row max first. Then the
largest exponent is exactly $e^0 = 1$ and overflow is impossible.

Two sanity checks worth remembering: uniform logits over $C$ classes give $\log C$ (that is your
"random guessing" baseline — 2.30 for 10 classes), and every row of the gradient sums to zero,
because probabilities sum to one.

### Signature
```python
def softmax_cross_entropy(logits, labels):
    # logits: (N, C) raw scores
    # labels: (N,) integer class indices
    # returns: (loss, dlogits) — scalar mean loss and its (N, C) gradient
    ...
```

### Rules
- Pure **NumPy** — no PyTorch
- Subtract the row max before `exp` (must survive logits of ±1000)
- Average the loss over the batch, and divide the gradient by `N` to match

### Example
```
logits = np.zeros((4, 10))
loss, dlogits = softmax_cross_entropy(logits, np.array([0, 3, 7, 9]))
# loss == log(10) == 2.302...,  dlogits.sum() == 0
```

In [ ]:
import numpy as np

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def softmax_cross_entropy(logits, labels):
    # logits: (N, C);  labels: (N,) int
    # returns (loss, dlogits)
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
np.random.seed(0)
logits = np.random.randn(6, 4)
labels = np.array([0, 1, 2, 3, 0, 1])

loss, dlogits = softmax_cross_entropy(logits, labels)
print("loss        :", float(loss))
print("dlogits     :", dlogits.shape)
print("rows sum ~0 :", np.allclose(dlogits.sum(axis=1), 0))

uniform_loss, _ = softmax_cross_entropy(np.zeros((4, 10)), np.array([0, 3, 7, 9]))
print("uniform loss:", float(uniform_loss), "== log(10) =", np.log(10))

big, _ = softmax_cross_entropy(np.array([[1000.0, 1001.0, 999.0]]), np.array([1]))
print("stable      :", np.isfinite(big))

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("numpy_softmax_ce")